In [1]:
# Notebook 1: VGG & InceptionV3 (CLASSIFICATION MODELS ONLY)

# Import essential libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import zipfile
import shutil
import cv2
import pickle
import json
from tqdm.notebook import tqdm
from google.colab import drive
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.decomposition import PCA
import gc
import psutil
import time


# TensorFlow and Keras imports
from tensorflow.keras.applications import VGG16, InceptionV3
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Flatten, GlobalAveragePooling2D,
    Add, Reshape, Conv1D, MaxPooling1D, BatchNormalization, Activation
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau



# Create directories for results
os.makedirs('notebook5/Notebook1Results', exist_ok=True)

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration parameters
CONFIG = {
    'image_size': (224, 224),        # Standard size for VGG and InceptionV3
    'batch_size': 128,               # Larger batch size for A100 GPU
    'angles': ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180'],  # All angles
    'scenarios': ['nm', 'bg', 'cl'],  # Normal, carrying bag, wearing coat
    'sample_ratio': 1,             # Use full dataset since extraction already completed
    'epochs': 32,                    # More epochs with early stopping
    'max_samples': None,             # No limit since feature extraction is done
    'results_dir': 'notebook5/Notebook1Results',
    'early_stopping_patience': 10,   # Increased patience
    'model_seed': 42                 # For reproducibility
}

# Class mapping for easy reference
CLASS_MAPPING = {
    'nm': 0,  # normal walking
    'bg': 1,  # carrying bag
    'cl': 2   # wearing coat
}

In [2]:
# ONLY ABSOLUTELY NECESSARY FUNCTIONS

def mount_drive():
    """Mount Google Drive to access dataset"""
    drive.mount('/content/drive')

def extract_dataset(zip_path, extract_to):
    """Extract dataset from zip file"""
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Extracting files"):
            if not file_info.filename.startswith('__MACOSX'):
                zip_ref.extract(file_info, extract_to)

    macosx_path = os.path.join(extract_to, '__MACOSX')
    if os.path.exists(macosx_path):
        shutil.rmtree(macosx_path)

    return extract_to

def is_valid_image(image_path):
    """Check if an image file is valid"""
    try:
        img = cv2.imread(image_path)
        if img is None or img.shape[2] != 3:
            return False
        return True
    except:
        return False

def get_walking_style_from_path(path):
    """Extract walking style from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part.startswith('nm-') or part == 'nm':
            return 'nm'
        elif part.startswith('bg-') or part == 'bg':
            return 'bg'
        elif part.startswith('cl-') or part == 'cl':
            return 'cl'
    return None

def get_angle_from_path(path):
    """Extract viewing angle from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part in CONFIG['angles']:
            return part
    return None

def build_dataset_index(base_path, sample_ratio=1.0):
    """Build an index of all valid image files"""
    dataset_index = []
    styles_count = {'nm': 0, 'bg': 0, 'cl': 0}
    angles_count = {angle: 0 for angle in CONFIG['angles']}

    # For visualization in Notebook 5: track detailed counts
    detailed_counts = {
        f"{style}_{angle}": 0
        for style in CONFIG['scenarios']
        for angle in CONFIG['angles']
    }

    for root, _, files in tqdm(list(os.walk(base_path)), desc="Scanning folders"):
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if sample_ratio < 1.0 and len(image_files) > 0:
            num_samples = max(1, int(len(image_files) * sample_ratio))
            image_files = np.random.choice(image_files, num_samples, replace=False)

        for file in image_files:
            file_path = os.path.join(root, file)
            walking_style = get_walking_style_from_path(file_path)
            angle = get_angle_from_path(file_path)
            if walking_style and angle and is_valid_image(file_path):
                styles_count[walking_style] += 1
                angles_count[angle] += 1
                # Update detailed counts for Notebook 5
                detailed_counts[f"{walking_style}_{angle}"] += 1

                dataset_index.append({
                    'path': file_path,
                    'style': walking_style,
                    'angle': angle,
                    'style_id': CLASS_MAPPING[walking_style]
                })

    df = pd.DataFrame(dataset_index)

    # Save distribution data for Notebook 5
    distribution_data = {
        'styles_count': styles_count,
        'angles_count': angles_count,
        'detailed_counts': detailed_counts,  # Added for Notebook 5
        'scenarios': CONFIG['scenarios'],
        'angles': CONFIG['angles']
    }
    with open(os.path.join(CONFIG['results_dir'], 'distribution_data.pkl'), 'wb') as f:
        pickle.dump(distribution_data, f)

    return df

def load_dataset_from_drive():
    """Load dataset from Google Drive"""
    mount_drive()
    zip_path = '/content/drive/MyDrive/CASIA B dataset.zip'
    extract_to = '/content/dataset'
    dataset_path = extract_dataset(zip_path, extract_to)
    dataset_df = build_dataset_index(dataset_path, sample_ratio=CONFIG['sample_ratio'])
    return dataset_df, dataset_path

In [3]:
def preprocess_image(image_path, target_size=CONFIG['image_size']):
    """Load and preprocess an image"""
    try:
        img = cv2.imread(image_path)
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        return img
    except:
        return np.zeros((*target_size, 3), dtype=np.float32)

def build_feature_extractors():
    """Build VGG and InceptionV3 feature extractor models"""
    models = {}

    # VGG16
    vgg_base = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in vgg_base.layers[:-8]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['vgg'] = Model(
        inputs=vgg_base.input,
        outputs=GlobalAveragePooling2D()(vgg_base.output),
        name='vgg_features'
    )

    # InceptionV3
    inception_base = InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in inception_base.layers[:-30]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['inception'] = Model(
        inputs=inception_base.input,
        outputs=GlobalAveragePooling2D()(inception_base.output),
        name='inception_features'
    )

    return models

def extract_batch_features(model, image_paths):
    """Extract features for a batch of images"""
    batch_size = len(image_paths)
    batch_images = np.zeros((batch_size, *CONFIG['image_size'], 3), dtype=np.float32)
    for i, path in enumerate(image_paths):
        batch_images[i] = preprocess_image(path)
    features = model.predict(batch_images, verbose=0)
    return features

def process_and_extract_features(models, dataset_df, max_samples=None):
    """Process the dataset and extract features"""
    if max_samples is not None and max_samples < len(dataset_df):
        dataset_df = dataset_df.sample(max_samples)

    total_samples = len(dataset_df)
    features = {model_name: [] for model_name in models.keys()}
    labels = []
    paths = []
    angles = []

    # For visualization in Notebook 5: track processing times
    processing_times = {
        'model': {model_name: [] for model_name in models.keys()},
        'angle': {angle: {model_name: [] for model_name in models.keys()} for angle in CONFIG['angles']}
    }

    batch_size = CONFIG['batch_size']
    for start_idx in tqdm(range(0, total_samples, batch_size), desc="Processing batches"):
        end_idx = min(start_idx + batch_size, total_samples)
        batch_df = dataset_df.iloc[start_idx:end_idx]

        batch_paths = batch_df['path'].values
        batch_labels = batch_df['style_id'].values
        batch_angles = batch_df['angle'].values

        for model_name, model in models.items():
            start_time = time.time()
            batch_features = extract_batch_features(model, batch_paths)
            end_time = time.time()

            # Track processing time for each model
            processing_time = end_time - start_time
            processing_times['model'][model_name].append(processing_time)

            # Track processing time by angle
            for angle in np.unique(batch_angles):
                angle_indices = np.where(batch_angles == angle)[0]
                if len(angle_indices) > 0:
                    avg_time_per_sample = processing_time / len(batch_paths)
                    angle_time = avg_time_per_sample * len(angle_indices)
                    processing_times['angle'][angle][model_name].append(angle_time)

            features[model_name].extend(batch_features)

        labels.extend(batch_labels)
        paths.extend(batch_paths)
        angles.extend(batch_angles)
        gc.collect()

    # Convert lists to numpy arrays
    for model_name in models.keys():
        features[model_name] = np.array(features[model_name])
    labels = np.array(labels)
    angles = np.array(angles)

    # Convert labels to one-hot encoding
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=len(CLASS_MAPPING))

    # Compute average processing times
    avg_times = {
        'model': {model_name: np.mean(times) for model_name, times in processing_times['model'].items()},
        'angle': {angle: {model_name: np.mean(times) if times else 0
                        for model_name, times in model_times.items()}
                for angle, model_times in processing_times['angle'].items()}
    }

    # Save features for Notebook 5
    features_data = {
        'features': features,
        'labels': labels_onehot,
        'raw_labels': labels,
        'paths': paths,
        'angles': angles,
        'processing_times': processing_times,  # Raw processing times
        'avg_processing_times': avg_times      # Average processing times
    }
    with open(os.path.join(CONFIG['results_dir'], 'features_data.pkl'), 'wb') as f:
        pickle.dump(features_data, f)

    return features, labels_onehot, paths, angles

def apply_pca_to_features(features, n_components=0.95):
    """Apply PCA to reduce feature dimensionality"""
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(features)

    # Calculate cumulative variance
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    # Create a safe function to find components for a threshold
    def find_components_for_threshold(threshold):
        indices = np.where(cumulative_variance >= threshold)[0]
        if len(indices) > 0:
            return int(indices[0]) + 1
        else:
            # If threshold is never reached, return the total number of components
            return len(cumulative_variance)

    # Save PCA data for Notebook 5
    pca_data = {
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'explained_variance': pca.explained_variance_,
        'n_components': pca.n_components_,
        'cumulative_variance': cumulative_variance,
        # Add components for specific variance thresholds for visualization
        'variance_thresholds': {
            '50%': find_components_for_threshold(0.5),
            '75%': find_components_for_threshold(0.75),
            '90%': find_components_for_threshold(0.9),
            '95%': find_components_for_threshold(0.95),
            '99%': find_components_for_threshold(0.99)
        }
    }
    with open(os.path.join(CONFIG['results_dir'], 'pca_data.pkl'), 'wb') as f:
        pickle.dump(pca_data, f)

    # Save the PCA model
    with open(os.path.join(CONFIG['results_dir'], 'pca_model.pkl'), 'wb') as f:
        pickle.dump(pca, f)

    return reduced_features, pca

def concatenate_features(feature_dict, apply_pca=True):
    """Concatenate features from multiple models"""
    concatenated = np.concatenate([feature_dict['vgg'], feature_dict['inception']], axis=1)

    if apply_pca:
        # Use more components to retain more information
        reduced_features, pca_model = apply_pca_to_features(concatenated, n_components=768)
        with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
            pickle.dump({'raw': concatenated, 'reduced': reduced_features}, f)
        return reduced_features, pca_model

    with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
        pickle.dump({'raw': concatenated}, f)
    return concatenated, None

In [4]:
# Enhanced Classification Models

# Enhanced CNN Classifier with your implementation
def build_enhanced_cnn_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build an enhanced CNN classifier with residual connections"""
    inputs = Input(shape=input_shape)

    # First dense block
    x = Dense(1024, activation='relu')(inputs)
    x = Dropout(0.4)(x)

    # Residual connection block 1
    res1 = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(res1)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = tf.keras.layers.add([res1, x])  # Residual connection

    # Residual connection block 2
    res2 = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(res2)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = tf.keras.layers.add([res2, x])  # Residual connection

    # Final dense layers with batch normalization
    x = Dense(128, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)

    # Create model
    model = Model(inputs=inputs, outputs=outputs, name="enhanced_cnn")

    # Compile model with better optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0003)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Enhanced LSTM Classifier with your implementation
def build_enhanced_lstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build LSTM classifier with a simpler architecture"""
    model = Sequential(name="lstm_classifier")

    # Reshape input for LSTM (creating a sequence of 1 timestep)
    model.add(tf.keras.layers.Reshape((1, input_shape[0]), input_shape=input_shape))

    # LSTM layers
    model.add(LSTM(256, return_sequences=False))
    model.add(Dropout(0.5))

    # Dense layers
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.4))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)  # Higher learning rate
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Enhanced 3D CNN Classifier with your implementation
def build_3dcnn_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a 3D CNN classifier that dynamically adjusts to any input shape"""
    model = Sequential(name="3dcnn_classifier")

    # Get the feature length
    feature_length = input_shape[0]

    # We'll create a more flexible approach - convert to a 1D temporal sequence first
    # Then use 1D convolutions instead of 3D
    model.add(tf.keras.layers.Reshape((feature_length, 1), input_shape=input_shape))

    # Use 1D convolutions instead
    model.add(tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(Dropout(0.4))

    model.add(tf.keras.layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(Dropout(0.4))

    # LSTM layer to capture temporal patterns
    model.add(LSTM(64))
    model.add(Dropout(0.3))

    # Dense layers
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# BiLSTM Classifier with your implementation
def build_bilstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a more advanced Bidirectional LSTM model"""
    model = Sequential(name="bilstm_classifier")

    # Reshape input for LSTM
    model.add(tf.keras.layers.Reshape((1, input_shape[0]), input_shape=input_shape))

    # Bidirectional LSTM layers
    model.add(tf.keras.layers.Bidirectional(LSTM(384, return_sequences=True)))
    model.add(Dropout(0.4))

    model.add(tf.keras.layers.Bidirectional(LSTM(192)))
    model.add(Dropout(0.3))

    # Dense layers with batch normalization for better generalization
    model.add(Dense(128, activation='relu'))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(Dropout(0.3))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    # Compile model with a medium learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# CNN-LSTM Classifier (keeping the original implementation)
def build_cnn_lstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a hybrid CNN-LSTM classifier"""
    # Reshape input for CNN part
    inputs = Input(shape=input_shape)
    reshaped = Reshape((input_shape[0], 1))(inputs)

    # CNN part
    x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(reshaped)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # LSTM part
    x = LSTM(384, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="cnn_lstm_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# CNN-BiLSTM Classifier (keeping the original implementation)
def build_cnn_bilstm_classifier(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a hybrid CNN-BiLSTM classifier"""
    # Reshape input for CNN part
    inputs = Input(shape=input_shape)
    reshaped = Reshape((input_shape[0], 1))(inputs)

    # CNN part
    x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(reshaped)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(256, kernel_size=3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # BiLSTM part
    x = tf.keras.layers.Bidirectional(LSTM(384, return_sequences=False))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name="cnn_bilstm_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [5]:
# Train model function with enhanced callbacks
def train_model(model, X_train, y_train, X_val, y_val, epochs=CONFIG['epochs'],
                batch_size=CONFIG['batch_size'], model_name="model"):
    """Train a model with enhanced callbacks"""
    # Use class weights to handle imbalance
    class_counts = np.sum(y_train, axis=0)
    total = np.sum(class_counts)
    class_weights = {i: total / (len(class_counts) * count) for i, count in enumerate(class_counts)}

    # Set patience based on model type
    patience = CONFIG['early_stopping_patience']

    # Enhanced callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=patience,
            restore_best_weights=True,
            verbose=1,
            min_delta=0.001  # Require more improvement for early stopping
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,          # More aggressive reduction
            patience=patience//2,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            os.path.join(CONFIG['results_dir'], f"{model_name}_best.h5"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    # Time tracking for training
    start_time = time.time()

    # Train model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )

    # Calculate training time
    training_time = time.time() - start_time

    # Save the model and history
    model.save(os.path.join(CONFIG['results_dir'], f"{model_name}_final.h5"))

    # Enhanced history data for visualization
    history_data = {
        'history': history.history,
        'training_time': training_time,
        'epochs_completed': len(history.history['loss']),
        'best_val_accuracy': max(history.history['val_accuracy']),
        'best_val_loss': min(history.history['val_loss']),
        'final_train_accuracy': history.history['accuracy'][-1],
        'final_train_loss': history.history['loss'][-1]
    }

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_history.pkl"), 'wb') as f:
        pickle.dump(history_data, f)

    return history, model, training_time

def evaluate_model(model, X_test, y_test, angles_test=None, model_name="model"):
    """Evaluate model and save results with handling for missing classes"""
    # Predict on test set
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # Calculate accuracy and confusion matrix
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Calculate precision, recall, and F1 score with zero_division=0
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    # Store all metrics
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm.tolist()
    }

    # Get unique classes in the data
    unique_classes = sorted(np.unique(np.concatenate([y_true, y_pred])))
    class_names = [list(CLASS_MAPPING.keys())[i] for i in unique_classes]

    # If we have fewer classes than expected, log this
    if len(unique_classes) < len(CLASS_MAPPING):
        print(f"Warning: Only detected {len(unique_classes)} classes in data/predictions: {class_names}")
        print(f"Expected {len(CLASS_MAPPING)} classes: {list(CLASS_MAPPING.keys())}")

    try:
        # Try with explicit labels parameter
        report = classification_report(
            y_true, y_pred,
            labels=unique_classes,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    except Exception as e:
        print(f"Warning: Could not generate classification report: {e}")
        # Fallback to a basic report as dictionary
        report = {
            'accuracy': accuracy,
            'weighted_precision': precision,
            'weighted_recall': recall,
            'weighted_f1': f1
        }

    # Combine metrics and report
    all_metrics = {**metrics, 'classification_report': report}

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_metrics.json"), 'w') as f:
        json.dump(all_metrics, f)

    # Save confusion matrix
    np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix.npy"), cm)

    # Calculate and save metrics by angle if angles are provided
    if angles_test is not None:
        angle_metrics = {}
        for angle in np.unique(angles_test):
            angle_idx = np.where(angles_test == angle)[0]

            # Per-angle predictions and ground truth
            angle_y_true = y_true[angle_idx]
            angle_y_pred = y_pred[angle_idx]

            # Skip if no samples for this angle
            if len(angle_y_true) == 0:
                continue

            # Calculate metrics
            angle_acc = accuracy_score(angle_y_true, angle_y_pred)

            try:
                angle_prec, angle_recall, angle_f1, _ = precision_recall_fscore_support(
                    angle_y_true, angle_y_pred, average='weighted', zero_division=0
                )
            except Exception:
                # Fallback if metrics can't be calculated
                angle_prec, angle_recall, angle_f1 = 0.0, 0.0, 0.0

            # Save confusion matrix for each angle (safely)
            try:
                angle_cm = confusion_matrix(angle_y_true, angle_y_pred)
                np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix_angle_{angle}.npy"), angle_cm)
            except Exception:
                angle_cm = [[0]]

            # Store all per-angle metrics
            angle_metrics[angle] = {
                'accuracy': float(angle_acc),
                'precision': float(angle_prec),
                'recall': float(angle_recall),
                'f1': float(angle_f1),
                'confusion_matrix': angle_cm.tolist() if hasattr(angle_cm, 'tolist') else angle_cm,
                'samples': int(len(angle_y_true))
            }

            # Get per-class metrics for this angle (safely)
            try:
                # Get unique classes in this angle's data
                angle_unique_classes = sorted(np.unique(np.concatenate([angle_y_true, angle_y_pred])))
                angle_class_names = [list(CLASS_MAPPING.keys())[i] for i in angle_unique_classes]

                angle_class_report = classification_report(
                    angle_y_true, angle_y_pred,
                    labels=angle_unique_classes,
                    target_names=angle_class_names,
                    output_dict=True,
                    zero_division=0
                )
                angle_metrics[angle]['classification_report'] = angle_class_report
            except Exception as e:
                print(f"Warning: Could not generate classification report for angle {angle}: {e}")
                # Just include basic metrics
                angle_metrics[angle]['classification_report'] = {
                    'accuracy': float(angle_acc),
                    'weighted_precision': float(angle_prec),
                    'weighted_recall': float(angle_recall),
                    'weighted_f1': float(angle_f1)
                }

        # Save angle metrics
        with open(os.path.join(CONFIG['results_dir'], f"{model_name}_angle_metrics.json"), 'w') as f:
            json.dump(angle_metrics, f)

    print(f"Model: {model_name}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    return accuracy, cm, metrics

In [6]:
def train_all_classification_models(concatenated_features, labels, angles, test_size=0.2, val_size=0.1):
    """Train all classification models and save results"""
    # Split data
    X_train_val, X_test, y_train_val, y_test, angles_train_val, angles_test = train_test_split(
        concatenated_features, labels, angles, test_size=test_size,
        stratify=np.argmax(labels, axis=1), random_state=42
    )

    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val, angles_train, angles_val = train_test_split(
        X_train_val, y_train_val, angles_train_val, test_size=val_ratio,
        stratify=np.argmax(y_train_val, axis=1), random_state=42
    )

    print(f"Data split: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")

    # Save split data for reproducibility
    split_data = {
        'X_train_shape': X_train.shape,
        'X_val_shape': X_val.shape,
        'X_test_shape': X_test.shape,
        'angles_train': angles_train,
        'angles_val': angles_val,
        'angles_test': angles_test,
        'class_distribution_train': np.sum(y_train, axis=0).tolist(),
        'class_distribution_val': np.sum(y_val, axis=0).tolist(),
        'class_distribution_test': np.sum(y_test, axis=0).tolist()
    }
    with open(os.path.join(CONFIG['results_dir'], 'data_split_info.pkl'), 'wb') as f:
        pickle.dump(split_data, f)

    # Enhanced models to train with specific seeds
    models_to_train = [
        ('Enhanced_CNN', build_enhanced_cnn_classifier(X_train.shape[1:]), 123),
        ('Enhanced_LSTM', build_enhanced_lstm_classifier(X_train.shape[1:]), 456),
        ('BiLSTM', build_bilstm_classifier(X_train.shape[1:]), 789),
        ('3DCNN', build_3dcnn_classifier(X_train.shape[1:]), 101),  # Using your 3DCNN implementation
        ('CNN_LSTM', build_cnn_lstm_classifier(X_train.shape[1:]), 202),
        ('CNN_BiLSTM', build_cnn_bilstm_classifier(X_train.shape[1:]), 303)
    ]

    # Train each model
    results = {}
    for model_name, model, seed in models_to_train:
        print(f"\n{'='*50}")
        print(f"Training {model_name} model with seed {seed}...")
        print(f"{'='*50}")

        tf.random.set_seed(seed)
        np.random.seed(seed)

        history, trained_model, training_time = train_model(
            model, X_train, y_train, X_val, y_val,
            epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'],
            model_name=model_name
        )

        print(f"\nEvaluating {model_name} model...")
        accuracy, cm, metrics = evaluate_model(
            trained_model, X_test, y_test, angles_test=angles_test, model_name=model_name
        )

        results[model_name] = {
            'model': trained_model,
            'history': history,
            'accuracy': accuracy,
            'confusion_matrix': cm,
            'metrics': metrics,
            'training_time': training_time
        }

        # Save important data separately
        with open(os.path.join(CONFIG['results_dir'], f"{model_name}_results.pkl"), 'wb') as f:
            pickle.dump({
                'accuracy': accuracy,
                'confusion_matrix': cm,
                'history': history.history,
                'metrics': metrics,
                'training_time': training_time
            }, f)

    # Create and save performance table with all metrics
    performance_metrics = []
    for name, info in results.items():
        metrics = info['metrics']
        performance_metrics.append({
            'Model': name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1'],
            'Training_Time': info['training_time']
        })

    performance_df = pd.DataFrame(performance_metrics).sort_values('Accuracy', ascending=False)
    performance_df.to_csv(os.path.join(CONFIG['results_dir'], "model_performance.csv"), index=False)

    print("\nModel Performance Ranking:")
    print(performance_df.to_string(index=False))

    # Save combined results
    with open(os.path.join(CONFIG['results_dir'], "all_classification_results.pkl"), 'wb') as f:
        pickle.dump(results, f)

    return results, performance_df

In [7]:
# Main execution function with fixed JSON serialization
def run_notebook1():
    """Run Notebook 1 pipeline: VGG & InceptionV3 for Classification with improvements"""
    print("="*80)
    print("NOTEBOOK 1: VGG & INCEPTIONV3 FEATURE EXTRACTION AND CLASSIFICATION MODELS")
    print("="*80)

    # Update configuration for better performance
    CONFIG.update({
        'image_size': (256, 256),        # Increased image size for better feature extraction
        'batch_size': 128,                # Smaller batch size for better generalization
        'sample_ratio': 1,             # Use full dataset
        'epochs': 32,                   # More epochs with early stopping
        'early_stopping_patience': 15,   # Increased patience for better convergence
        'max_samples': None

    })

    print(f"Using enhanced configuration: batch_size={CONFIG['batch_size']}, image_size={CONFIG['image_size']}")

    # Create a file to track the start time
    start_time = time.time()
    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump({'notebook1_start_time': start_time}, f)

    # Helper function to make values JSON serializable
    def make_serializable(obj):
        """Convert numpy types to Python native types for JSON serialization"""
        if isinstance(obj, (np.integer, np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return [make_serializable(item) for item in obj]
        elif isinstance(obj, dict):
            return {k: make_serializable(v) for k, v in obj.items()}
        else:
            return obj

    # Load dataset
    print("\nLoading dataset...")
    dataset_df, dataset_path = load_dataset_from_drive()
    dataset_load_time = time.time()

    # Build feature extractors - keeping VGG & InceptionV3 but with improved parameters
    print("\nBuilding feature extractors...")
    feature_extractors = build_feature_extractors()
    feature_extractor_build_time = time.time()

    # Extract features with data augmentation
    print("\nExtracting features with augmentation...")
    # Add data augmentation to preprocess_image
    def preprocess_image_with_aug(image_path, target_size=CONFIG['image_size'], training=True):
        """Load and preprocess an image with optional augmentation for training"""
        try:
            # Load and resize image
            img = cv2.imread(image_path)
            img = cv2.resize(img, target_size)

            # Convert BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Apply data augmentation if training
            if training:
                # Random horizontal flip with 50% probability
                if np.random.random() > 0.5:
                    img = cv2.flip(img, 1)

                # Random brightness adjustment
                brightness = np.random.uniform(0.8, 1.2)
                img = np.clip(img * brightness, 0, 255).astype(np.uint8)

                # Random rotation (-10 to +10 degrees)
                angle = np.random.uniform(-10, 10)
                h, w = img.shape[:2]
                M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
                img = cv2.warpAffine(img, M, (w, h))

            # Normalize pixel values
            img = img.astype(np.float32) / 255.0

            return img
        except Exception as e:
            print(f"Error preprocessing image {image_path}: {e}")
            # Return blank image as fallback
            return np.zeros((*target_size, 3), dtype=np.float32)

    # Modify the extract_batch_features function to use our enhanced preprocessing
    def extract_batch_features_enhanced(model, image_paths):
        """Extract features for a batch of images with enhanced preprocessing"""
        batch_size = len(image_paths)
        batch_images = np.zeros((batch_size, *CONFIG['image_size'], 3), dtype=np.float32)

        for i, path in enumerate(image_paths):
            batch_images[i] = preprocess_image_with_aug(path)

        features = model.predict(batch_images, verbose=0)
        return features

    # Store the original function
    original_extract_batch = globals().get('extract_batch_features', None)

    # Replace with enhanced version
    globals()['extract_batch_features'] = extract_batch_features_enhanced

    features, labels, paths, angles = process_and_extract_features(
        feature_extractors, dataset_df, max_samples=CONFIG['max_samples']
    )

    # Restore original function if it existed
    if original_extract_batch:
        globals()['extract_batch_features'] = original_extract_batch

    feature_extraction_time = time.time()

    # Concatenate features with PCA - using optimized parameters
    print("\nConcatenating features with PCA...")
    concatenated_features, pca_model = concatenate_features(features, apply_pca=True)

    # Save feature dimensions and properties for analysis - convert to regular Python types
    feature_info = {
        'vgg_dim': int(features['vgg'].shape[1]),
        'inception_dim': int(features['inception'].shape[1]),
        'raw_concatenated_dim': int(features['vgg'].shape[1] + features['inception'].shape[1]),
        'pca_reduced_dim': int(concatenated_features.shape[1]),
        'num_samples': int(concatenated_features.shape[0]),
    }

    # Add variance retained if available, safely converting to regular float
    if hasattr(pca_model, 'explained_variance_ratio_'):
        variance_sum = float(sum(pca_model.explained_variance_ratio_))
        feature_info['variance_retained'] = variance_sum

    with open(os.path.join(CONFIG['results_dir'], 'feature_info.json'), 'w') as f:
        json.dump(feature_info, f)

    pca_time = time.time()

    # Train classification models with improved parameters
    print("\nTraining classification models with enhanced parameters...")
    # Create a modified training function that uses improved parameters but keeps original model structure
    def train_models_with_enhancements(concatenated_features, labels, angles, test_size=0.15, val_size=0.15):
        """Train all classification models with improved training parameters"""
        # Split data with improved ratios (more validation data, less test data)
        X_train_val, X_test, y_train_val, y_test, angles_train_val, angles_test = train_test_split(
            concatenated_features, labels, angles, test_size=test_size,
            stratify=np.argmax(labels, axis=1), random_state=42
        )

        val_ratio = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val, angles_train, angles_val = train_test_split(
            X_train_val, y_train_val, angles_train_val, test_size=val_ratio,
            stratify=np.argmax(y_train_val, axis=1), random_state=42
        )

        print(f"Data split: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")

        # Convert NumPy arrays to Python types for serialization
        split_data = {
            'X_train_shape': [int(dim) for dim in X_train.shape],
            'X_val_shape': [int(dim) for dim in X_val.shape],
            'X_test_shape': [int(dim) for dim in X_test.shape],
            'class_distribution_train': [float(count) for count in np.sum(y_train, axis=0)],
            'class_distribution_val': [float(count) for count in np.sum(y_val, axis=0)],
            'class_distribution_test': [float(count) for count in np.sum(y_test, axis=0)]
        }

        # Safely convert angles data
        if hasattr(angles_train, 'tolist'):
            split_data['angles_train'] = angles_train.tolist()
        else:
            split_data['angles_train'] = make_serializable(angles_train)

        if hasattr(angles_val, 'tolist'):
            split_data['angles_val'] = angles_val.tolist()
        else:
            split_data['angles_val'] = make_serializable(angles_val)

        if hasattr(angles_test, 'tolist'):
            split_data['angles_test'] = angles_test.tolist()
        else:
            split_data['angles_test'] = make_serializable(angles_test)

        with open(os.path.join(CONFIG['results_dir'], 'data_split_info.pkl'), 'wb') as f:
            pickle.dump(split_data, f)

        # Enhanced models to train with specific seeds
        models_to_train = [
            ('Enhanced_CNN', build_enhanced_cnn_classifier(X_train.shape[1:]), 123),
            ('Enhanced_LSTM', build_enhanced_lstm_classifier(X_train.shape[1:]), 456),
            ('BiLSTM', build_bilstm_classifier(X_train.shape[1:]), 789),
            ('3DCNN', build_3dcnn_classifier(X_train.shape[1:]), 101),
            ('CNN_LSTM', build_cnn_lstm_classifier(X_train.shape[1:]), 202),
            ('CNN_BiLSTM', build_cnn_bilstm_classifier(X_train.shape[1:]), 303)
        ]

        # Train each model
        results = {}
        for model_name, model, seed in models_to_train:
            print(f"\n{'='*50}")
            print(f"Training {model_name} model with seed {seed}...")
            print(f"{'='*50}")

            tf.random.set_seed(seed)
            np.random.seed(seed)

            # Use enhanced callbacks
            callbacks = [
                EarlyStopping(
                    monitor='val_accuracy',
                    patience=CONFIG['early_stopping_patience'],
                    restore_best_weights=True,
                    verbose=1,
                    min_delta=0.001
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.3,
                    patience=CONFIG['early_stopping_patience']//2,
                    min_lr=1e-6,
                    verbose=1
                ),
                ModelCheckpoint(
                    os.path.join(CONFIG['results_dir'], f"{model_name}_best.h5"),
                    monitor='val_accuracy',
                    save_best_only=True,
                    verbose=1
                )
            ]

            # Class weights for imbalanced data
            class_counts = np.sum(y_train, axis=0)
            total = np.sum(class_counts)
            class_weights = {int(i): float(total / (len(class_counts) * count)) for i, count in enumerate(class_counts)}

            # Train model with advanced technique
            start_time = time.time()
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                batch_size=CONFIG['batch_size'],
                epochs=CONFIG['epochs'],
                callbacks=callbacks,
                class_weight=class_weights,
                verbose=1
            )
            training_time = time.time() - start_time

            # Save the model and history
            model.save(os.path.join(CONFIG['results_dir'], f"{model_name}_final.h5"))

            # Enhanced history data - convert NumPy values to Python types
            history_data = {
                'training_time': float(training_time),
                'epochs_completed': int(len(history.history['loss'])),
                'best_val_accuracy': float(max(history.history['val_accuracy'])),
                'best_val_loss': float(min(history.history['val_loss'])),
                'final_train_accuracy': float(history.history['accuracy'][-1]),
                'final_train_loss': float(history.history['loss'][-1])
            }

            # Convert history data to Python types
            history_data['history'] = {
                k: [float(v) for v in vals] for k, vals in history.history.items()
            }

            with open(os.path.join(CONFIG['results_dir'], f"{model_name}_history.pkl"), 'wb') as f:
                pickle.dump(history_data, f)

            # Evaluate with original function
            print(f"\nEvaluating {model_name} model...")
            accuracy, cm, metrics = evaluate_model(
                model, X_test, y_test, angles_test=angles_test, model_name=model_name
            )

            # Process metrics to ensure JSON serializability
            metrics_json_safe = make_serializable(metrics)

            results[model_name] = {
                'model': model,
                'history': history,
                'accuracy': float(accuracy),
                'confusion_matrix': make_serializable(cm),
                'metrics': metrics_json_safe,
                'training_time': float(training_time)
            }

            # Save important data separately
            with open(os.path.join(CONFIG['results_dir'], f"{model_name}_results.pkl"), 'wb') as f:
                pickle.dump({
                    'accuracy': float(accuracy),
                    'confusion_matrix': make_serializable(cm),
                    'history': history_data['history'],
                    'metrics': metrics_json_safe,
                    'training_time': float(training_time)
                }, f)

        # Create and save performance table with all metrics
        performance_metrics = []
        for name, info in results.items():
            metrics = info['metrics']
            performance_metrics.append({
                'Model': name,
                'Accuracy': float(metrics['accuracy']),
                'Precision': float(metrics['precision']),
                'Recall': float(metrics['recall']),
                'F1': float(metrics['f1']),
                'Training_Time': float(info['training_time'])
            })

        performance_df = pd.DataFrame(performance_metrics).sort_values('Accuracy', ascending=False)
        performance_df.to_csv(os.path.join(CONFIG['results_dir'], "model_performance.csv"), index=False)

        print("\nModel Performance Ranking:")
        print(performance_df.to_string(index=False))

        # Save combined results
        with open(os.path.join(CONFIG['results_dir'], "all_classification_results.pkl"), 'wb') as f:
            pickle.dump(make_serializable(results), f)

        return results, performance_df

    # Use our enhanced training function while keeping the structure same
    model_results, performance_df = train_models_with_enhancements(
        concatenated_features, labels, angles
    )
    model_training_time = time.time()

    # Save best model info
    best_model_name = performance_df.iloc[0]['Model']
    best_model_accuracy = float(performance_df.iloc[0]['Accuracy'])

    # Save execution times - ensure all values are regular Python types
    execution_times = {
        'total_time': float(model_training_time - start_time),
        'dataset_loading_time': float(dataset_load_time - start_time),
        'feature_extractor_build_time': float(feature_extractor_build_time - dataset_load_time),
        'feature_extraction_time': float(feature_extraction_time - feature_extractor_build_time),
        'pca_time': float(pca_time - feature_extraction_time),
        'model_training_time': float(model_training_time - pca_time),
        'models': {name: float(info['training_time']) for name, info in model_results.items()}
    }

    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump(execution_times, f)

    print(f"\nBest model: {best_model_name} with accuracy {best_model_accuracy:.4f}")
    print(f"\nAll results saved to: {CONFIG['results_dir']}")

    # Create README with results summary
    with open(os.path.join(CONFIG['results_dir'], "README.md"), 'w') as f:
        f.write("# Notebook 1: VGG & InceptionV3 Classification Results (Enhanced)\n\n")
        f.write("## Feature Extraction\n")
        f.write(f"- VGG16 feature dimension: {features['vgg'].shape[1]}\n")
        f.write(f"- InceptionV3 feature dimension: {features['inception'].shape[1]}\n")
        f.write(f"- Concatenated feature dimension: {concatenated_features.shape[1]}\n\n")
        f.write("## Classification Models Performance\n")
        f.write(performance_df.to_markdown(index=False) + "\n\n")
        f.write(f"### Best Model: {best_model_name}\n")
        f.write(f"- Accuracy: {best_model_accuracy:.4f}\n")
        f.write(f"- Total execution time: {execution_times['total_time']:.2f} seconds\n")
        f.write("\n## Enhancement Details\n")
        f.write("- Increased image size to 256x256\n")
        f.write("- Added data augmentation during feature extraction\n")
        f.write("- Used improved training parameters\n")
        f.write("- Enhanced model architectures\n")

    # Create a summary of data collected for Notebook 5
    notebook5_data_summary = {
        'distribution_data': 'Sample distribution by walking style and angle',
        'features_data': 'Extracted features, labels, processing times',
        'pca_data': 'PCA variance ratios, components, thresholds',
        'concatenated_features': 'Raw and reduced features',
        'execution_times': 'Timing of each pipeline component',
        'model_metrics': 'Performance metrics for each model',
        'model_history': 'Training history for each model',
        'confusion_matrices': 'Overall and per-angle confusion matrices',
        'angle_metrics': 'Detailed performance by viewing angle',
        'enhancements': 'Details of enhancements made to improve performance'
    }

    # Save summary for easier reference
    with open(os.path.join(CONFIG['results_dir'], 'data_summary_for_notebook5.json'), 'w') as f:
        json.dump(notebook5_data_summary, f)

    print("\nNOTEBOOK 1 COMPLETED (ENHANCED VERSION)")
    return {
        'feature_extractors': feature_extractors,
        'features': features,
        'concatenated_features': concatenated_features,
        'pca_model': pca_model,
        'labels': labels,
        'angles': angles,
        'paths': paths,
        'models': model_results,
        'performance': performance_df,
        'execution_times': execution_times
    }

# Run the notebook
if __name__ == "__main__":
    run_notebook1()

NOTEBOOK 1: VGG & INCEPTIONV3 FEATURE EXTRACTION AND CLASSIFICATION MODELS
Using enhanced configuration: batch_size=128, image_size=(256, 256)

Loading dataset...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



Building feature extractors...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Extracting features with augmentation...



Concatenating features with PCA...

Training classification models with enhanced parameters...
Data split: Train=396470, Val=84958, Test=84958


/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Enhanced_CNN model with seed 123...
Epoch 1/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5869 - loss: 0.9138
Epoch 1: val_accuracy improved from -inf to 0.74795, saving model to notebook5/Notebook1Results/Enhanced_CNN_best.h5


...

3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6154 - loss: 0.8518 - val_accuracy: 0.6255 - val_loss: 0.8409 - learning_rate: 3.0000e-04
Epoch 12/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6200 - loss: 0.8441
Epoch 12: val_accuracy did not improve from 0.62549
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6200 - loss: 0.8441 - val_accuracy: 0.5608 - val_loss: 0.9354 - learning_rate: 3.0000e-04
Epoch 13/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6243 - loss: 0.8352
Epoch 13: val_accuracy did not improve from 0.62549
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6243 - loss: 0.8352 - val_accuracy: 0.6120 - val_loss: 0.8511 - learning_rate: 3.0000e-04
Epoch 14/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6277 - loss: 0.8282
Epoch 14: val_accuracy improved from 0.62549 to 0.64265, saving model to notebook5/Notebook1Results/CNN_BiLSTM_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6277 - loss: 0.8282 - val_accuracy: 0.6426 - val_loss: 0.8079 - learning_rate: 3.0000e-04
Epoch 15/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6334 - loss: 0.8180
Epoch 15: val_accuracy did not improve from 0.64265
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 33ms/step - accuracy: 0.6334 - loss: 0.8180 - val_accuracy: 0.6422 - val_loss: 0.8039 - learning_rate: 3.0000e-04
Epoch 16/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6383 - loss: 0.8090
Epoch 16: val_accuracy improved from 0.64265 to 0.64668, saving model to notebook5/Notebook1Results/CNN_BiLSTM_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6383 - loss: 0.8090 - val_accuracy: 0.6467 - val_loss: 0.7967 - learning_rate: 3.0000e-04
Epoch 17/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6424 - loss: 0.8001
Epoch 17: val_accuracy did not improve from 0.64668
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6424 - loss: 0.8001 - val_accuracy: 0.6059 - val_loss: 0.8621 - learning_rate: 3.0000e-04
Epoch 18/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6464 - loss: 0.7892
Epoch 18: val_accuracy improved from 0.64668 to 0.66013, saving model to notebook5/Notebook1Results/CNN_BiLSTM_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6464 - loss: 0.7892 - val_accuracy: 0.6601 - val_loss: 0.7683 - learning_rate: 3.0000e-04
Epoch 19/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6511 - loss: 0.7805
Epoch 19: val_accuracy did not improve from 0.66013
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6511 - loss: 0.7805 - val_accuracy: 0.6556 - val_loss: 0.7784 - learning_rate: 3.0000e-04
Epoch 20/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6549 - loss: 0.7713
Epoch 20: val_accuracy did not improve from 0.66013
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6549 - loss: 0.7713 - val_accuracy: 0.6460 - val_loss: 0.7896 - learning_rate: 3.0000e-04
Epoch 21/32
3097/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6572 - loss: 0.7614
Epoch 21: val_accuracy did not improve from 0.66013
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.6572 - loss: 0.7614 - val_accuracy: 0.6390 - val_loss: 0.8024 - 


Evaluating CNN_BiLSTM model...
2655/2655 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step
Model: CNN_BiLSTM, Accuracy: 0.6606, Precision: 0.6951, Recall: 0.6606, F1: 0.6712
Confusion Matrix:
[[35709  8503  7339]
 [ 4476  9251  2918]
 [ 2793  2804 11165]]

Model Performance Ranking:
        Model  Accuracy  Precision   Recall       F1  Training_Time
       BiLSTM  0.833506   0.834906 0.833506 0.834131    1068.902913
 Enhanced_CNN  0.830528   0.830110 0.830528 0.830287     359.492505
Enhanced_LSTM  0.810836   0.818132 0.810836 0.813362     554.122412
     CNN_LSTM  0.673368   0.716539 0.673368 0.685928    2266.583687
   CNN_BiLSTM  0.660621   0.695149 0.660621 0.671248    3323.353742
        3DCNN  0.581417   0.589586 0.581417 0.583940    1344.945470

Best model: BiLSTM with accuracy 0.8335

All results saved to: notebook5/Notebook1Results

NOTEBOOK 1 COMPLETED (ENHANCED VERSION)
